# WildFyre: historical fire-persistence readiness

This source-only notebook was imported from the teammate's LinuxONE notebook.
See `docs/ml-readiness.md` for provenance, reported metrics and known limits.
Stored outputs were removed. Downloading and training are **off by default**.

The target is a further qualifying detection in the same approximate 5 km cell
in the **24 hours after the current six-hour window closes**. Only already-active
cell-windows are modelled. This is detection persistence, not spread, animal risk
or fire ETA. Features are available at window close, not window start.

Train on June–October 2022/2023; use 2024 only for chronological evaluation.
The original timestamp-unit and incomplete-horizon bugs are corrected here.
**Regenerate labels and metrics before trusting any handed-over model.**


## 1. Verify the LinuxONE ML environment

In [ ]:
import platform
import sys

import numpy as np
import pandas as pd
import sklearn
import joblib

print("Python:", sys.version)
print("Architecture:", platform.machine())
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Joblib:", joblib.__version__)

In [ ]:
try:
    import xgboost as xgb
    print("XGBoost:", xgb.__version__)
    print("XGBoost available")
except Exception as exc:
    print("XGBoost unavailable:")
    print(exc)

try:
    import tensorflow as tf
    print("TensorFlow:", tf.__version__)
    print("TensorFlow available")
except Exception as exc:
    print("TensorFlow unavailable:")
    print(exc)

## 2. Verify shared storage between JupyterLab and the LinuxONE host

In [ ]:
import os
from pathlib import Path

SHARED_DIR = Path(os.environ.get("WILDFYRE_SHARED_DIR", "/home/ibm-user/shared"))
if not SHARED_DIR.is_dir():
    raise FileNotFoundError("Shared storage is unavailable; set WILDFYRE_SHARED_DIR.")
print("Shared directory:", SHARED_DIR)


In [ ]:
test_path = SHARED_DIR / "wildfyre_jupyter_test.txt"

test_path.write_text(
    "WildFyre JupyterLab is successfully writing to LinuxONE shared storage."
)

print("Created:", test_path)
print("Host-side equivalent: /home/linux1/shared/wildfyre_jupyter_test.txt")

## 3. Project configuration

In [ ]:
PROJECT_NAME = "WildFyre"
REGION_NAME = "California"

# Current rule-based prototype radius.
PROXIMITY_RADIUS_KM = 25

# Historical modelling design.
GRID_SIZE_KM = 5.0
OBSERVATION_WINDOW_HOURS = 6
PREDICTION_HORIZON_HOURS = 24

print("Project:", PROJECT_NAME)
print("Region:", REGION_NAME)
print("Proximity radius:", PROXIMITY_RADIUS_KM, "km")
print("Grid size:", GRID_SIZE_KM, "km")
print("Observation window:", OBSERVATION_WINDOW_HOURS, "hours")
print("Prediction horizon:", PREDICTION_HORIZON_HOURS, "hours")

In [ ]:
DATA_DIR = SHARED_DIR / "data"
# Corrected outputs never overwrite the handed-over dataset/model artifacts.
OUTPUT_DIR = SHARED_DIR / "wf07-corrected"
MODEL_DIR = OUTPUT_DIR / "models"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
print("Input directory:", DATA_DIR)
print("Corrected output directory:", OUTPUT_DIR)


In [ ]:
data_files = sorted(DATA_DIR.glob("*"))

if not data_files:
    print("No WildFyre datasets have been added yet.")
else:
    print("Available datasets:")
    for file in data_files:
        print("-", file.name)

## 4. General helper functions

In [ ]:
def load_csv(filename):
    path = DATA_DIR / filename

    if not path.exists():
        raise FileNotFoundError(f"Dataset not found: {path}")

    df = pd.read_csv(path)

    print("Loaded:", filename)
    print("Rows:", len(df))
    print("Columns:", len(df.columns))

    return df

In [ ]:
def inspect_dataset(df, rows=5):
    print("Shape:")
    print(df.shape)

    print("\nColumns:")
    print(df.columns.tolist())

    print("\nData types:")
    print(df.dtypes)

    print("\nMissing values:")
    print(df.isna().sum())

    print("\nFirst rows:")
    display(df.head(rows))

In [ ]:
def valid_coordinates(latitude, longitude):
    return (
        pd.notna(latitude)
        and pd.notna(longitude)
        and -90 <= latitude <= 90
        and -180 <= longitude <= 180
    )

print(valid_coordinates(34.2, -118.5))
print(valid_coordinates(200, -118.5))

### Great-circle distance helper used by the existing rule-based prototype

In [ ]:
from math import radians, sin, cos, sqrt, atan2

EARTH_RADIUS_KM = 6371.0088

def distance_km(lat1, lon1, lat2, lon2):
    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        sin(dlat / 2) ** 2
        + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    )

    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    return EARTH_RADIUS_KM * c

print("Distance for 1 degree at the equator:", distance_km(0, 0, 0, 1), "km")

In [ ]:
def proximity_score(distance):
    return max(
        0.0,
        1.0 - distance / PROXIMITY_RADIUS_KM
    )

for d in [0, 5, 12.5, 25, 30]:
    print(f"{d:5.1f} km -> {proximity_score(d):.3f}")

## 5. Historical NASA FIRMS data

### Source choice

For historical modelling, use **NASA FIRMS VIIRS S-NPP Standard Processing** (`VIIRS_SNPP_SP`) rather than presenting near-real-time detections as a historical science dataset.

### Demo region

California bounding box:

- West: -124.5
- South: 32.5
- East: -114.0
- North: 42.1

The FIRMS map key is entered interactively and is **not written into the notebook**.

In [ ]:
from datetime import timedelta
from io import StringIO
import getpass
import time
import requests

WEST = -124.5
SOUTH = 32.5
EAST = -114.0
NORTH = 42.1

BBOX = f"{WEST},{SOUTH},{EAST},{NORTH}"
FIRMS_SOURCE = "VIIRS_SNPP_SP"

print("FIRMS source:", FIRMS_SOURCE)
print("Bounding box:", BBOX)

### Enter the FIRMS API key securely

In [ ]:
DOWNLOAD_HISTORICAL_DATA = os.environ.get("WILDFYRE_DOWNLOAD_HISTORICAL") == "1"
FIRMS_MAP_KEY = ""
if DOWNLOAD_HISTORICAL_DATA:
    FIRMS_MAP_KEY = getpass.getpass("FIRMS map key (not saved): ").strip()
    if not FIRMS_MAP_KEY:
        raise ValueError("A FIRMS map key is required for downloading.")
print("Historical download enabled:", DOWNLOAD_HISTORICAL_DATA)


### Optional connection check

Runs only with `WILDFYRE_DOWNLOAD_HISTORICAL=1`. HTTP failures are sanitized
because a FIRMS request URL includes the map key. Cached-data runs need no key.


In [ ]:
if DOWNLOAD_HISTORICAL_DATA:
    test_url = (
        "https://firms.modaps.eosdis.nasa.gov/api/area/csv/"
        f"{FIRMS_MAP_KEY}/{FIRMS_SOURCE}/{BBOX}/5/2024-07-01"
    )
    try:
        response = requests.get(test_url, timeout=(3.05, 30))
        response.raise_for_status()
    except requests.RequestException:
        raise RuntimeError("FIRMS connection check failed; request details withheld.") from None
    print("FIRMS connection check succeeded.")


In [ ]:
if DOWNLOAD_HISTORICAL_DATA:
    try:
        test_fires = pd.read_csv(StringIO(response.text))
    except pd.errors.EmptyDataError:
        test_fires = pd.DataFrame()
    print("Connection-check rows:", len(test_fires))


### Reusable historical download function

In [ ]:
from pathlib import Path
from io import StringIO
import time
import pandas as pd
import requests


CHUNK_CACHE_DIR = DATA_DIR / "firms_chunks" / FIRMS_SOURCE / BBOX.replace(",", "_")
CHUNK_CACHE_DIR.mkdir(parents=True, exist_ok=True)


def fetch_firms_period(
    start_date,
    end_date,
    chunk_days=2,
    pause_seconds=0.5,
    max_retries=4,
):
    """
    Download FIRMS data in small cached chunks.

    Each successful chunk is saved immediately.
    If the notebook is interrupted, rerunning skips
    chunks that have already been downloaded.
    """

    if not FIRMS_MAP_KEY:
        raise ValueError("FIRMS_MAP_KEY has not been loaded.")

    start = pd.Timestamp(start_date)
    end = pd.Timestamp(end_date)

    if end < start:
        raise ValueError("end_date must be on or after start_date.")

    year_cache = CHUNK_CACHE_DIR / str(start.year)
    year_cache.mkdir(parents=True, exist_ok=True)

    chunks = []
    current = start

    while current <= end:
        remaining_days = (end - current).days + 1
        day_range = min(chunk_days, remaining_days)

        date_string = current.strftime("%Y-%m-%d")

        cache_file = (
            year_cache
            / f"{date_string}_{day_range}d.csv"
        )

        # Use cached chunk if already downloaded
        if cache_file.exists():
            print(f"Cached: {date_string} ({day_range} days)")

            try:
                chunk = pd.read_csv(cache_file)

                if not chunk.empty:
                    chunks.append(chunk)

            except pd.errors.EmptyDataError:
                pass

            current += pd.Timedelta(days=day_range)
            continue

        url = (
            "https://firms.modaps.eosdis.nasa.gov/api/area/csv/"
            f"{FIRMS_MAP_KEY}/"
            f"{FIRMS_SOURCE}/"
            f"{BBOX}/"
            f"{day_range}/"
            f"{date_string}"
        )

        print(
            f"Downloading {date_string} "
            f"({day_range} days)..."
        )

        response = None

        for attempt in range(1, max_retries + 1):
            try:
                response = requests.get(
                    url,
                    timeout=(5, 30),
                )

                response.raise_for_status()
                break

            except requests.exceptions.RequestException as exc:
                if attempt == max_retries:
                    raise RuntimeError("Historical FIRMS download failed; request details withheld.") from None

                wait_seconds = attempt * 3

                print(
                    f"  {type(exc).__name__}. "
                    f"Retrying in {wait_seconds}s "
                    f"({attempt}/{max_retries})..."
                )

                time.sleep(wait_seconds)

        try:
            chunk = pd.read_csv(StringIO(response.text))

        except pd.errors.EmptyDataError:
            chunk = pd.DataFrame()

        # Save immediately so progress survives interruption
        if not chunk.empty:
            chunk.to_csv(cache_file, index=False)
            chunks.append(chunk)

            print(f"  Saved {len(chunk):,} rows")
        else:
            print("  No detections returned")

        current += pd.Timedelta(days=day_range)
        time.sleep(pause_seconds)

    if not chunks:
        return pd.DataFrame()

    return pd.concat(chunks, ignore_index=True)


## 6. Load historical seasons

Provide the three original `firms_california_<year>_raw.csv` files in DATA_DIR.
Set `WILDFYRE_DOWNLOAD_HISTORICAL=1` before starting the kernel only when a new
download is intended. Existing raw files must have the documented product,
bounds and complete season coverage; row timestamps alone cannot prove coverage.
Do not reuse a partial or unverified raw CSV as a complete season.


In [ ]:
FIRE_SEASONS = {
    2022: ("2022-06-01", "2022-10-31"),
    2023: ("2023-06-01", "2023-10-31"),
    2024: ("2024-06-01", "2024-10-31"),
}

print("Historical download enabled:", DOWNLOAD_HISTORICAL_DATA)
print("Configured seasons:", FIRE_SEASONS)


In [ ]:
raw_by_year = {}

for year, (start_date, end_date) in FIRE_SEASONS.items():
    raw_path = DATA_DIR / f"firms_california_{year}_raw.csv"

    if raw_path.exists():
        print(f"{year}: loading cached file {raw_path.name}")
        raw_by_year[year] = pd.read_csv(raw_path)
        continue

    if not DOWNLOAD_HISTORICAL_DATA:
        print(
            f"{year}: no cached file. "
            "Set DOWNLOAD_HISTORICAL_DATA = True to download it."
        )
        continue

    print(f"\nDownloading {year} fire season...")
    df_year = fetch_firms_period(start_date, end_date)

    df_year.to_csv(raw_path, index=False)
    raw_by_year[year] = df_year

    print(f"{year}: saved {len(df_year):,} rows -> {raw_path}")

In [ ]:
for year, df_year in raw_by_year.items():
    print(year, df_year.shape)

missing_years = sorted(set(FIRE_SEASONS) - set(raw_by_year))
if missing_years:
    raise FileNotFoundError(f"Missing complete raw FIRMS seasons: {missing_years}. See docs/ml-readiness.md.")


## 7. Clean and validate FIRMS detections

In [ ]:
def clean_firms_data(df):
    if df.empty:
        return df.copy()

    required_columns = {
        "latitude",
        "longitude",
        "acq_date",
        "acq_time",
        "satellite",
        "confidence",
    }

    missing = required_columns - set(df.columns)
    if missing:
        raise ValueError(f"Missing required FIRMS columns: {sorted(missing)}")

    cleaned = df.copy()

    numeric_columns = [
        "latitude",
        "longitude",
        "bright_ti4",
        "bright_ti5",
        "frp",
        "scan",
        "track",
    ]

    for column in numeric_columns:
        if column in cleaned.columns:
            cleaned[column] = pd.to_numeric(
                cleaned[column],
                errors="coerce"
            )

    # Valid Earth coordinates.
    cleaned = cleaned[
        cleaned["latitude"].between(-90, 90)
        & cleaned["longitude"].between(-180, 180)
    ].copy()

    # Keep records within the configured California demo bounding box.
    cleaned = cleaned[
        cleaned["latitude"].between(SOUTH, NORTH)
        & cleaned["longitude"].between(WEST, EAST)
    ].copy()

    # FIRMS acquisition time is HHMM UTC. Convert robustly to a four-character string.
    acq_time = (
        cleaned["acq_time"]
        .astype(str)
        .str.replace(r"\.0$", "", regex=True)
        .where(lambda values: values.str.fullmatch(r"\d{1,4}", na=False))
        .str.zfill(4)
    )

    cleaned["observed_at"] = pd.to_datetime(
        cleaned["acq_date"].astype(str)
        + " "
        + acq_time.str[:2]
        + ":"
        + acq_time.str[2:4],
        errors="coerce",
        utc=True
    )

    cleaned = cleaned.dropna(subset=["observed_at"])

    # FRP may be missing, but negative non-null FRP is rejected.
    if "frp" in cleaned.columns:
        cleaned = cleaned[
            cleaned["frp"].isna()
            | (np.isfinite(cleaned["frp"]) & (cleaned["frp"] >= 0))
        ].copy()

    duplicate_subset = [
        column
        for column in [
            "latitude",
            "longitude",
            "observed_at",
            "satellite",
        ]
        if column in cleaned.columns
    ]

    cleaned = cleaned.drop_duplicates(
        subset=duplicate_subset
    )

    return (
        cleaned
        .sort_values("observed_at")
        .reset_index(drop=True)
    )

In [ ]:
clean_by_year = {}

for year, df_year in raw_by_year.items():
    cleaned = clean_firms_data(df_year)

    print(
        f"{year}: raw={len(df_year):,}, "
        f"clean={len(cleaned):,}"
    )

    clean_by_year[year] = cleaned

    clean_path = OUTPUT_DIR / f"firms_california_{year}_clean.csv"
    cleaned.to_csv(clean_path, index=False)

    print("Saved:", clean_path)


### Inspect confidence and day/night values before filtering

In [ ]:
for year, df_year in clean_by_year.items():
    print(f"\n===== {year} =====")

    if "confidence" in df_year.columns:
        print("Confidence:")
        print(df_year["confidence"].value_counts(dropna=False))

    if "daynight" in df_year.columns:
        print("\nDay/night:")
        print(df_year["daynight"].value_counts(dropna=False))

### Keep normal/high-confidence detections

For the first prototype we retain VIIRS detections whose confidence category is `n` (nominal/normal) or `h` (high). The unfiltered cleaned files remain saved separately so this decision is reversible.

In [ ]:
analysis_by_year = {}

for year, df_year in clean_by_year.items():
    if "confidence" not in df_year.columns:
        analysis_by_year[year] = df_year.copy()
        continue

    confidence = (
        df_year["confidence"]
        .astype(str)
        .str.lower()
        .str.strip()
    )

    filtered = df_year[
        confidence.isin(["n", "h"])
    ].copy()

    analysis_by_year[year] = filtered

    print(
        f"{year}: {len(filtered):,} normal/high-confidence "
        f"rows retained from {len(df_year):,}"
    )

In [ ]:
for year, df_year in analysis_by_year.items():
    print(f"\n===== {year} type =====")

    if "type" in df_year.columns:
        print(
            df_year["type"]
            .value_counts(dropna=False)
            .sort_index()
        )

In [ ]:
fire_by_year = {}

for year, df_year in analysis_by_year.items():
    if "type" not in df_year.columns:
        raise ValueError(
            f"FIRMS type column missing for {year}"
        )

    filtered = df_year[
        pd.to_numeric(
            df_year["type"],
            errors="coerce"
        ) == 0
    ].copy()

    fire_by_year[year] = filtered

    print(
        f"{year}: {len(filtered):,} presumed vegetation-fire "
        f"rows retained from {len(df_year):,}"
    )

## 8. Combine seasons and inspect data quality

In [ ]:
if fire_by_year:
    combined_fires = pd.concat(
        [
            df.assign(season_year=year)
            for year, df in sorted(fire_by_year.items())
        ],
        ignore_index=True
    )

    combined_fires["observed_at"] = pd.to_datetime(
        combined_fires["observed_at"],
        utc=True,
        errors="coerce"
    )

    combined_fires = (
        combined_fires
        .dropna(subset=["observed_at"])
        .sort_values("observed_at")
        .reset_index(drop=True)
    )

    combined_path = (
        OUTPUT_DIR
        / "firms_california_2022_2024_vegetation_fire.csv"
    )

    combined_fires.to_csv(
        combined_path,
        index=False
    )

    print("Combined rows:", len(combined_fires))
    print("Saved:", combined_path)

    display(combined_fires.head())

else:
    combined_fires = pd.DataFrame()

    print(
        "No filtered vegetation-fire seasons are loaded."
    )


In [ ]:
if not combined_fires.empty:
    summary = (
        combined_fires
        .groupby("season_year")
        .agg(
            rows=("observed_at", "size"),
            start=("observed_at", "min"),
            end=("observed_at", "max"),
            unique_days=(
                "observed_at",
                lambda x: x.dt.date.nunique()
            ),
            mean_frp=("frp", "mean"),
            max_frp=("frp", "max"),
        )
    )

    display(summary)

    print("\nMissing values:")

    display(
        combined_fires[
            [
                column
                for column in [
                    "latitude",
                    "longitude",
                    "frp",
                    "bright_ti4",
                    "bright_ti5",
                    "confidence",
                    "daynight",
                    "type",
                ]
                if column in combined_fires.columns
            ]
        ]
        .isna()
        .sum()
        .to_frame("missing")
    )

    if "type" in combined_fires.columns:
        print("\nFIRMS type distribution:")

        display(
            combined_fires["type"]
            .value_counts(dropna=False)
            .sort_index()
            .to_frame("rows")
        )

## 9. Convert point detections into 5 km × 6-hour modelling windows

The model is conditional on a cell already having an active-fire detection in the current observation window.

We use an approximate local grid for this prototype:

- north/south distance: ~111.32 km per degree latitude
- east/west scale adjusted using the midpoint latitude of the California bounding box

This is sufficient for a hackathon prototype but should be replaced with a proper projected coordinate system for operational geospatial work.

In [ ]:
KM_PER_DEG_LAT = 111.32
REFERENCE_LAT = (SOUTH + NORTH) / 2
KM_PER_DEG_LON = 111.32 * np.cos(np.deg2rad(REFERENCE_LAT))

print("Reference latitude:", REFERENCE_LAT)
print("Approx km/degree longitude:", KM_PER_DEG_LON)

In [ ]:
def add_grid_and_time_window(df):
    if df.empty:
        return df.copy()

    out = df.copy()

    out["grid_x"] = np.floor(
        ((out["longitude"] - WEST) * KM_PER_DEG_LON)
        / GRID_SIZE_KM
    ).astype("int64")

    out["grid_y"] = np.floor(
        ((out["latitude"] - SOUTH) * KM_PER_DEG_LAT)
        / GRID_SIZE_KM
    ).astype("int64")

    out["window_start"] = out["observed_at"].dt.floor(
        f"{OBSERVATION_WINDOW_HOURS}h"
    )

    return out

In [ ]:
if not combined_fires.empty:
    gridded_fires = add_grid_and_time_window(combined_fires)

    print("Rows:", len(gridded_fires))
    print(
        "Unique grid cells:",
        gridded_fires[["grid_x", "grid_y"]]
        .drop_duplicates()
        .shape[0]
    )

    display(
        gridded_fires[
            [
                "latitude",
                "longitude",
                "observed_at",
                "grid_x",
                "grid_y",
                "window_start",
                "frp",
            ]
        ].head()
    )
else:
    gridded_fires = pd.DataFrame()

### Aggregate each active grid-cell window into model features

In [ ]:
def aggregate_fire_windows(df):
    if df.empty:
        return pd.DataFrame()

    work = df.copy()

    # Indicator features.
    confidence = (
        work["confidence"]
        .astype(str)
        .str.lower()
        .str.strip()
    )
    daynight = (
        work["daynight"]
        .astype(str)
        .str.upper()
        .str.strip()
        if "daynight" in work.columns
        else pd.Series("", index=work.index)
    )

    work["high_confidence_detection"] = (confidence == "h").astype(int)
    work["normal_confidence_detection"] = (confidence == "n").astype(int)
    work["day_detection"] = (daynight == "D").astype(int)
    work["night_detection"] = (daynight == "N").astype(int)

    agg_map = {
        "observed_at": "size",
        "latitude": "mean",
        "longitude": "mean",
        "high_confidence_detection": "sum",
        "normal_confidence_detection": "sum",
        "day_detection": "sum",
        "night_detection": "sum",
    }

    if "frp" in work.columns:
        agg_map["frp"] = ["mean", "max"]

    if "bright_ti4" in work.columns:
        agg_map["bright_ti4"] = ["mean", "max"]

    if "bright_ti5" in work.columns:
        agg_map["bright_ti5"] = ["mean", "max"]

    grouped = work.groupby(
        ["season_year", "grid_x", "grid_y", "window_start"],
        as_index=False
    ).agg(agg_map)

    # Flatten MultiIndex column names created by multiple aggregations.
    flattened = []
    for col in grouped.columns:
        if isinstance(col, tuple):
            name = "_".join(
                str(part)
                for part in col
                if part not in ("", None)
            )
        else:
            name = str(col)
        flattened.append(name)

    grouped.columns = flattened

    rename_map = {
        "observed_at_size": "fire_count",
        "latitude_mean": "cell_mean_latitude",
        "longitude_mean": "cell_mean_longitude",
        "high_confidence_detection_sum": "high_confidence_count",
        "normal_confidence_detection_sum": "normal_confidence_count",
        "day_detection_sum": "day_detection_count",
        "night_detection_sum": "night_detection_count",
        "frp_mean": "mean_frp",
        "frp_max": "max_frp",
        "bright_ti4_mean": "mean_bright_ti4",
        "bright_ti4_max": "max_bright_ti4",
        "bright_ti5_mean": "mean_bright_ti5",
        "bright_ti5_max": "max_bright_ti5",
    }

    grouped = grouped.rename(columns=rename_map)

    grouped["month"] = grouped["window_start"].dt.month
    grouped["hour_utc"] = grouped["window_start"].dt.hour

    return grouped.sort_values(
        ["season_year", "grid_x", "grid_y", "window_start"]
    ).reset_index(drop=True)

In [ ]:
fire_windows = aggregate_fire_windows(gridded_fires)

print("Aggregated fire windows:", len(fire_windows))

if not fire_windows.empty:
    display(fire_windows.head())

## 10. Create the 24-hour persistence label

For a current window `[t, t+6h)`, features become available at `t+6h`.
The four following bins start at `t+6h`, `t+12h`, `t+18h`, `t+24h` and cover
`[t+6h, t+30h)`. A further active bin in that set gives target 1; otherwise 0.
The bin at `t+30h` is outside the horizon. Satellite non-detection is not proof
that no fire exists. Season coverage must extend through `t+30h`.


In [ ]:
def add_persistence_target(
    windows,
    horizon_hours=PREDICTION_HORIZON_HOURS
):
    if windows.empty:
        return windows.copy()

    labelled_parts = []

    for (_, grid_x, grid_y), group in windows.groupby(
        ["season_year", "grid_x", "grid_y"],
        sort=False
    ):
        group = (
            group
            .sort_values("window_start")
            .copy()
        )

        times_ns = (
            group["window_start"]
            .dt.as_unit("ns")
            .astype("int64")
            .to_numpy()
        )

        horizon_ns = pd.Timedelta(
            hours=horizon_hours
        ).value

        targets = np.zeros(len(group), dtype=np.int8)

        for i, current_time in enumerate(times_ns):
            # Find the first strictly later active window.
            j = np.searchsorted(
                times_ns,
                current_time,
                side="right"
            )

            if j < len(times_ns):
                targets[i] = int(
                    times_ns[j] <= current_time + horizon_ns
                )

        group["target_persists_24h"] = targets
        labelled_parts.append(group)

    labelled = pd.concat(
        labelled_parts,
        ignore_index=True
    )

    return labelled.sort_values(
        ["season_year", "grid_x", "grid_y", "window_start"]
    ).reset_index(drop=True)


In [ ]:
def censor_incomplete_horizons(windows, season_ends):
    """season_ends maps each season to its verified exclusive coverage end."""
    out = windows.copy()
    if out.empty:
        return out
    out["prediction_at"] = out["window_start"] + pd.Timedelta(hours=OBSERVATION_WINDOW_HOURS)
    out["horizon_end"] = out["prediction_at"] + pd.Timedelta(hours=PREDICTION_HORIZON_HOURS)
    out["season_observation_end"] = out["season_year"].map(season_ends)
    if out["season_observation_end"].isna().any():
        raise ValueError("Missing season coverage; do not label unobserved horizons negative.")
    return out[out["horizon_end"] <= out["season_observation_end"]].reset_index(drop=True)

# Explicit confirmation is required: presence of a CSV does not prove that
# every requested day was retrieved successfully. Audit the raw cache first.
if os.environ.get("WILDFYRE_COMPLETE_SEASONS_CONFIRMED") != "1":
    raise RuntimeError("Audit full-season input coverage, then set WILDFYRE_COMPLETE_SEASONS_CONFIRMED=1.")
season_ends = {
    year: pd.Timestamp(end, tz="UTC") + pd.Timedelta(days=1)
    for year, (_, end) in FIRE_SEASONS.items()
}
labelled_windows = censor_incomplete_horizons(add_persistence_target(fire_windows), season_ends)
print("Corrected labelled windows:", len(labelled_windows))
print(labelled_windows["target_persists_24h"].value_counts().sort_index())


## 11. Save the prepared modelling dataset

This is the first dataset suitable for ML-readiness analysis. Weather features are **not added yet**.

In [ ]:
PREPARED_DATASET_PATH = (
    OUTPUT_DIR
    / "fire_persistence_windows_2022_2024.csv"
)

if not labelled_windows.empty:
    labelled_windows.to_csv(
        PREPARED_DATASET_PATH,
        index=False
    )

    print("Saved prepared dataset:")
    print(PREPARED_DATASET_PATH)
else:
    print("No labelled data available to save yet.")


## 12. Define the candidate feature set

In [ ]:
candidate_features = [
    "fire_count",
    "mean_frp",
    "max_frp",
    "mean_bright_ti4",
    "max_bright_ti4",
    "mean_bright_ti5",
    "max_bright_ti5",
    "high_confidence_count",
    "normal_confidence_count",
    "day_detection_count",
    "night_detection_count",
    "month",
    "hour_utc",
]

available_features = [
    feature
    for feature in candidate_features
    if feature in labelled_windows.columns
]

missing_candidate_features = [
    feature
    for feature in candidate_features
    if feature not in labelled_windows.columns
]

print("Available features:")
for feature in available_features:
    print("-", feature)

if missing_candidate_features:
    print("\nUnavailable candidate features:")
    for feature in missing_candidate_features:
        print("-", feature)

if missing_candidate_features:
    raise ValueError(f"Required model features unavailable: {missing_candidate_features}")
if np.isinf(labelled_windows[available_features].to_numpy(dtype=float, na_value=np.nan)).any():
    raise ValueError("Nonfinite feature values must be corrected before training.")


## 13. Chronological train/test split

Do **not** randomly split individual rows.

The initial split is:

- train: 2022 + 2023
- test: 2024

This reduces the chance that observations from the same continuing fire period leak into both train and test.

In [ ]:
if not labelled_windows.empty:
    train_df = labelled_windows[
        labelled_windows["season_year"].isin([2022, 2023])
    ].copy()

    test_df = labelled_windows[
        labelled_windows["season_year"] == 2024
    ].copy()

    print("Training windows:", len(train_df))
    print("Test windows:", len(test_df))

    if len(train_df):
        print(
            "Train positive rate:",
            train_df["target_persists_24h"].mean()
        )

    if len(test_df):
        print(
            "Test positive rate:",
            test_df["target_persists_24h"].mean()
        )
else:
    train_df = pd.DataFrame()
    test_df = pd.DataFrame()
    print("Prepared data is not available yet.")

## 14. ML-readiness checks

This section decides whether we are ready to move from data preparation into model training.

A `PASS` here is only a notebook-level readiness signal. Before claiming a trained WildFyre AI model, also document:

- data provenance and licensing
- prediction target and horizon
- leakage limitations
- baseline choice
- evaluation metrics
- whether weather or other point-in-time features are needed

In [ ]:
def build_ml_readiness_report():
    report = {
        "linuxone_s390x": platform.machine() == "s390x",
        "shared_storage_available": SHARED_DIR.exists(),
        "historical_data_loaded": not combined_fires.empty,
        "prepared_windows_available": not labelled_windows.empty,
        "target_defined": "target_persists_24h" in labelled_windows.columns,
        "prediction_horizon_defined": PREDICTION_HORIZON_HOURS == 24,
        "chronological_train_split_available": len(train_df) > 0,
        "chronological_test_split_available": len(test_df) > 0,
        "both_target_classes_in_train": (
            train_df["target_persists_24h"].nunique() == 2
            if len(train_df)
            else False
        ),
        "both_target_classes_in_test": (
            test_df["target_persists_24h"].nunique() == 2
            if len(test_df)
            else False
        ),
        "weather_features_added": False,
        "baseline_evaluation_completed": False,
        "model_evaluation_completed": False,
    }

    return report

ml_readiness = build_ml_readiness_report()

for item, ready in ml_readiness.items():
    print(f"{item}: {ready}")

In [ ]:
required_for_initial_training = [
    "linuxone_s390x",
    "shared_storage_available",
    "historical_data_loaded",
    "prepared_windows_available",
    "target_defined",
    "prediction_horizon_defined",
    "chronological_train_split_available",
    "chronological_test_split_available",
    "both_target_classes_in_train",
    "both_target_classes_in_test",
]

initial_training_ready = all(
    ml_readiness[item]
    for item in required_for_initial_training
)

print(
    "Initial FIRMS-only model training readiness:",
    "PASS" if initial_training_ready else "NOT READY"
)


In [ ]:
year_target_summary = (
    labelled_windows
    .groupby("season_year")["target_persists_24h"]
    .agg(
        samples="size",
        positives="sum",
        positive_rate="mean"
    )
)

year_target_summary["positive_rate"] = (
    year_target_summary["positive_rate"] * 100
).round(2)

display(year_target_summary)

## 15. Optional training (WF08 review required)

Training defaults to off. After review of the corrected labels, split and
acceptance criteria, set `WILDFYRE_TRAIN_MODEL=1` before starting the kernel.
The following cells then compare a dummy baseline and Random Forest; they do
not implement production inference or certify a calibrated probability.


In [ ]:
TRAIN_MODEL = os.environ.get("WILDFYRE_TRAIN_MODEL") == "1"
print("Model training enabled:", TRAIN_MODEL)


In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)

def validate_training_features(training, features):
    empty = [name for name in features if training[name].isna().all()]
    if empty:
        raise ValueError(f"Training features have no observed values: {empty}. Median imputation is undefined.")


In [ ]:
if TRAIN_MODEL:
    if not initial_training_ready:
        raise RuntimeError(
            "ML readiness checks have not passed. "
            "Do not train the model yet."
        )

    validate_training_features(train_df, available_features)
    X_train = train_df[available_features]
    y_train = train_df["target_persists_24h"]

    X_test = test_df[available_features]
    y_test = test_df["target_persists_24h"]

    baseline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("classifier", DummyClassifier(strategy="most_frequent")),
    ])

    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=300,
                random_state=42,
                n_jobs=-1,
                class_weight="balanced",
            ),
        ),
    ])

    baseline.fit(X_train, y_train)
    model.fit(X_train, y_train)

    print("Models trained.")
else:
    print(
        "Training skipped. "
        "Review the prepared dataset and readiness checks first."
    )

In [ ]:
def evaluate_classifier(name, estimator, X_test, y_test):
    predictions = estimator.predict(X_test)

    print(f"===== {name} =====")
    print("Accuracy:", accuracy_score(y_test, predictions))
    print(
        "Precision:",
        precision_score(
            y_test,
            predictions,
            zero_division=0
        )
    )
    print(
        "Recall:",
        recall_score(
            y_test,
            predictions,
            zero_division=0
        )
    )
    print(
        "F1:",
        f1_score(
            y_test,
            predictions,
            zero_division=0
        )
    )

    if hasattr(estimator, "predict_proba"):
        probabilities = estimator.predict_proba(X_test)[:, 1]

        if y_test.nunique() == 2:
            print("ROC AUC:", roc_auc_score(y_test, probabilities))

    print("\nConfusion matrix:")
    print(confusion_matrix(y_test, predictions))

    print("\nClassification report:")
    print(
        classification_report(
            y_test,
            predictions,
            zero_division=0
        )
    )

In [ ]:
if TRAIN_MODEL:
    evaluate_classifier(
        "Most-frequent baseline",
        baseline,
        X_test,
        y_test
    )

    evaluate_classifier(
        "Random Forest",
        model,
        X_test,
        y_test
    )
else:
    print("Evaluation skipped because TRAIN_MODEL is False.")

In [ ]:
if TRAIN_MODEL:
    feature_importance = pd.DataFrame({
        "feature": available_features,
        "importance": model.named_steps["classifier"].feature_importances_,
    }).sort_values(
        "importance",
        ascending=False
    ).reset_index(drop=True)

    display(feature_importance)


In [ ]:
if TRAIN_MODEL:
    from sklearn.inspection import permutation_importance

    permutation_result = permutation_importance(
        model,
        X_test,
        y_test,
        scoring="roc_auc",
        n_repeats=10,
        random_state=42,
        n_jobs=-1,
    )

    permutation_table = pd.DataFrame({
        "feature": available_features,
        "importance_mean": permutation_result.importances_mean,
        "importance_std": permutation_result.importances_std,
    }).sort_values(
        "importance_mean",
        ascending=False
    ).reset_index(drop=True)

    display(permutation_table)


In [ ]:
if TRAIN_MODEL:
    features_without_month = [
        feature
        for feature in available_features
        if feature != "month"
    ]

    X_train_no_month = train_df[features_without_month]
    X_test_no_month = test_df[features_without_month]

    model_no_month = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=300,
                random_state=42,
                n_jobs=-1,
                class_weight="balanced",
            ),
        ),
    ])

    model_no_month.fit(
        X_train_no_month,
        y_train
    )

    evaluate_classifier(
        "Random Forest without month",
        model_no_month,
        X_test_no_month,
        y_test
    )


## 16. Model saving helper

In [ ]:
def save_model(model, filename="fire_risk_model.joblib"):
    model_path = MODEL_DIR / filename

    joblib.dump(model, model_path)

    print("Saved model:")
    print(model_path)

    return model_path

In [ ]:
if TRAIN_MODEL:
    MODEL_PATH = save_model(
        model,
        "fire_persistence_random_forest.joblib"
    )

    feature_path = MODEL_DIR / "fire_persistence_features.txt"
    feature_path.write_text(
        "\n".join(available_features)
    )

    print("Saved feature order:", feature_path)
else:
    print("No model artifact saved.")

## 17. Next data enhancement: historical weather

The FIRMS-only dataset describes the current observed fire state. The next useful enhancement is to join historical, point-in-time weather features such as:

- temperature
- relative humidity
- wind speed
- wind direction
- precipitation

Weather must be aligned by location and timestamp without using information that would only be known after the prediction time.

Do not add synthetic weather values.

## 18. Final notebook status

In [ ]:
print("WildFyre ML notebook status")
print("--------------------------------")
print("Architecture:", platform.machine())
print("Shared storage:", SHARED_DIR.exists())
print("Data directory:", DATA_DIR.exists())
print("Model directory:", MODEL_DIR.exists())
print("Historical rows:", len(combined_fires))
print("Prepared windows:", len(labelled_windows))
print("Initial training ready:", initial_training_ready)
print("Training enabled:", TRAIN_MODEL)

print("\nCurrent phase: corrected ML readiness; historical rerun and WF08 review required.")


In [ ]:
if TRAIN_MODEL:
    import joblib
    import numpy as np

    loaded_model = joblib.load(MODEL_PATH)

    loaded_predictions = loaded_model.predict(X_test)

    print(
        "Reloaded predictions match:",
        np.array_equal(
            loaded_predictions,
            model.predict(X_test)
        )
    )


In [ ]:
import platform
import sklearn
import joblib
import numpy
import scipy

print("Architecture:", platform.machine())
print("Python:", platform.python_version())
print("scikit-learn:", sklearn.__version__)
print("joblib:", joblib.__version__)
print("numpy:", numpy.__version__)
print("scipy:", scipy.__version__)